In [2]:
import pandas as pd
import numpy as np

In [3]:
asset_pool = pd.read_csv('./funds_allocation/asset_pool_per_fund.csv')
fund_asset = pd.read_csv('./funds_allocation/fund_asset_allocation.csv')

In [4]:
fund_asset.head()

,FUND_ID,ASSET_CLASS_ID,PERCENT_OF_FUND
0,3,1,1.00
1,1,1,1.00
2,2,1,1.00
3,6,1,0.50
4,4,1,0.96


In [5]:
asset_pool.head()

,FUND_ID,ASSET_CLASS_ID,AVAILABLE_AMOUNT
0,9,1,1.192803e+09
1,9,2,2.071685e+09
2,9,3,3.080218e+09
3,9,5,1.142653e+09
4,9,6,1.339256e+09


In [6]:
merged_df = pd.merge(fund_asset, asset_pool, on=['FUND_ID', 'ASSET_CLASS_ID'])
merged_df
merged_df.sort_values(by=['PERCENT_OF_FUND'], ascending=False, inplace=True)
funds_aum = pd.DataFrame()
for fund_id, group in merged_df.groupby('FUND_ID'):
    aum = 1e17
    for _, row in group.iterrows():
        aum = row['AVAILABLE_AMOUNT']
        for __, irow in group.iterrows():
            if(irow['PERCENT_OF_FUND']==0):
                continue
            if(irow['PERCENT_OF_FUND'] * aum > row['AVAILABLE_AMOUNT']):
                break
    print(fund_id, aum)
    for _, row in group.iterrows():
        funds_aum = funds_aum._append(pd.DataFrame({"fund_id":[fund_id],"aum":[aum],"asset_class_id":[row['ASSET_CLASS_ID']],"amount_inv":[aum*row['PERCENT_OF_FUND']],"available_amt":[row['AVAILABLE_AMOUNT']],"PERCENT_OF_FUND":[row['PERCENT_OF_FUND']]}))
funds_aum

7 490795137.6524985
9 1142653082.3531566
10 469746022.6788497
11 1790428251.4877484
12 1528559151.7523959
13 469746022.6788497
14 1528559151.7523959
15 2136896580.3005788


,fund_id,aum,asset_class_id,amount_inv,available_amt,PERCENT_OF_FUND
0,7,4.907951e+08,2.0,2.650294e+08,8.548472e+08,0.54
0,7,4.907951e+08,3.0,1.815942e+08,1.273867e+09,0.37
0,7,4.907951e+08,8.0,3.926361e+07,1.031903e+09,0.08
0,7,4.907951e+08,7.0,0.000000e+00,4.734385e+08,0.00
0,7,4.907951e+08,4.0,0.000000e+00,1.265596e+09,0.00
...,...,...,...,...,...,...
0,15,2.136897e+09,7.0,1.709517e+08,7.588956e+08,0.08
0,15,2.136897e+09,8.0,8.547586e+07,1.728064e+09,0.04
0,15,2.136897e+09,1.0,6.410690e+07,7.987363e+08,0.03
0,15,2.136897e+09,6.0,4.273793e+07,9.045148e+08,0.02


In [7]:
funds_aum.drop_duplicates('fund_id')[['fund_id','aum']]

,fund_id,aum
0,7,4.907951e+08
0,9,1.142653e+09
0,10,4.697460e+08
0,11,1.790428e+09
0,12,1.528559e+09
0,13,4.697460e+08
0,14,1.528559e+09
0,15,2.136897e+09


In [8]:
perc_captured = pd.DataFrame(funds_aum['amount_inv']/funds_aum['available_amt'])
funds_aum['perc_captured']=perc_captured
funds_aum

,fund_id,aum,asset_class_id,amount_inv,available_amt,PERCENT_OF_FUND,perc_captured
0,7,4.907951e+08,2.0,2.650294e+08,8.548472e+08,0.54,0.310031
0,7,4.907951e+08,3.0,1.815942e+08,1.273867e+09,0.37,0.142554
0,7,4.907951e+08,8.0,3.926361e+07,1.031903e+09,0.08,0.038050
0,7,4.907951e+08,7.0,0.000000e+00,4.734385e+08,0.00,0.000000
0,7,4.907951e+08,4.0,0.000000e+00,1.265596e+09,0.00,0.000000
...,...,...,...,...,...,...,...
0,15,2.136897e+09,7.0,1.709517e+08,7.588956e+08,0.08,0.225264
0,15,2.136897e+09,8.0,8.547586e+07,1.728064e+09,0.04,0.049463
0,15,2.136897e+09,1.0,6.410690e+07,7.987363e+08,0.03,0.080260
0,15,2.136897e+09,6.0,4.273793e+07,9.045148e+08,0.02,0.047250


In [9]:
fund_df = pd.DataFrame()
for fund_id, group in funds_aum[funds_aum['amount_inv']>0].groupby('fund_id'):
    amt_inv = group['amount_inv'].sum()
    avl_inv = group['available_amt'].sum()
    fund_df = fund_df._append(pd.DataFrame({'fund_id':[fund_id],'perc_alloc':[amt_inv/avl_inv],'avl_aum':avl_inv,'amt_inv':[amt_inv]}))
fund_df
    

,fund_id,perc_alloc,avl_aum,amt_inv
0,7,0.153732,3.160617e+09,4.858872e+08
0,9,0.129245,8.841019e+09,1.142653e+09
0,10,0.106128,4.426213e+09,4.697460e+08
0,11,0.101313,1.749549e+10,1.772524e+09
0,12,0.106360,1.437163e+10,1.528559e+09
0,13,0.095535,4.917008e+09,4.697460e+08
0,14,0.073394,2.082664e+10,1.528559e+09
0,15,0.250462,8.531813e+09,2.136897e+09


In [19]:
! pip install pulp
import pulp as pulp
def get_best_possible_aum(available_amounts,percent_of_fund,fund_id):
    print(available_amounts)
    print(percent_of_fund)
    prob = pulp.LpProblem("Maximize_Total_Investment", pulp.LpMaximize)

# Define the decision variable Z (total investable amount)
    Z = pulp.LpVariable("Total_Investable_Amount", lowBound=0, cat="Continuous")

    # Constraints: Each individual investable amount (a_i * Z) must be less than or equal to the available amount
    for i in available_amounts.keys():
        prob += percent_of_fund[i] * Z <= available_amounts[i], f"Max_Investment_in_Asset_Class_{i}"

    # Objective function: Maximize Z
    prob += Z, "Maximize_Total_Investment"

    # Solve the problem
    prob.solve()
    
    results = {
        "FUND_ID" : [],
        "ASSET_CLASS_ID": [],
        "MAX_FEASIBLE_AUM": []
    }
    total_investment = pulp.value(prob.objective)
    
    # Populate the results dictionary with investable amounts per asset class
    for i in available_amounts.keys():
        investable_amount = percent_of_fund[i] * total_investment
        results['FUND_ID'].append(fund_id)
        results["ASSET_CLASS_ID"].append(i)
        results["MAX_FEASIBLE_AUM"].append(investable_amount)
    
    # Convert the results dictionary to a DataFrame
    result_df = pd.DataFrame(results)
    print(result_df)
    
    return result_df, total_investment



You should consider upgrading via the 'C:\Users\mohit\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip' command.


In [46]:
fund_asset.head()
final_df = pd.DataFrame()
asset_level_df = pd.DataFrame()
asset_pool['AVAILABLE_AMOUNT'] = asset_pool['AVAILABLE_AMOUNT'].astype(float)
fund_asset['PERCENT_OF_FUND'] = fund_asset['PERCENT_OF_FUND'].astype(float)
for fund_id in fund_asset['FUND_ID'].unique():
    if len(asset_pool[asset_pool['FUND_ID'] == fund_id].set_index('ASSET_CLASS_ID')['AVAILABLE_AMOUNT']) <= 0:
        continue
    percent_of_fund = fund_asset[fund_asset['FUND_ID'] == fund_id].set_index('ASSET_CLASS_ID')['PERCENT_OF_FUND'].astype(float).to_dict()
    available_amount = asset_pool[asset_pool['FUND_ID'] == fund_id].set_index('ASSET_CLASS_ID')['AVAILABLE_AMOUNT'].astype(float).to_dict()
    result_df, total_investment = get_best_possible_aum(available_amount,percent_of_fund,fund_id)
    result_df = result_df.merge(
    asset_pool[asset_pool['FUND_ID'] == fund_id][['ASSET_CLASS_ID', 'AVAILABLE_AMOUNT']],
    how='left',
    left_on='ASSET_CLASS_ID',  # Use the column name from result_df
    right_on='ASSET_CLASS_ID'
    ) 
    result_df = result_df.merge(
    fund_asset[fund_asset['FUND_ID'] == fund_id].set_index('ASSET_CLASS_ID')['PERCENT_OF_FUND'],
    how='left',
    left_on='ASSET_CLASS_ID',  # Use the column name from result_df
    right_on='ASSET_CLASS_ID'
    ).drop(columns=['ASSET_CLASS_ID']) 
    asset_level_df = asset_level_df._append(result_df)
    final_df = final_df._append(pd.DataFrame({'FUND_ID' : [fund_id], 'MAX_FEASIBLE_AUM' : [total_investment]}))
final_df.head()

{1: 1192802629.4759405, 2: 2071684512.8172495, 3: 3080217858.177684, 5: 1142653082.3531566, 6: 1339255985.986064, 7: 1151764561.1025603, 8: 2496313757.8445826, 4: 3060619206.6727}
{1: 0.09, 2: 0.81, 3: 0.08, 4: 0.0, 5: 0.0, 6: 0.0, 7: 0.0, 8: 0.02}
   FUND_ID  ASSET_CLASS_ID  MAX_FEASIBLE_AUM
0        9               1      2.301872e+08
1        9               2      2.071685e+09
2        9               3      2.046108e+08
3        9               5      0.000000e+00
4        9               6      0.000000e+00
5        9               7      0.000000e+00
6        9               8      5.115270e+07
7        9               4      0.000000e+00
{1: 490795137.6524985, 2: 854847163.6211282, 5: 469746022.6788497, 7: 473438512.5966285, 3: 1273866694.6678357, 4: 1265595962.5185652, 8: 1031903058.35007, 6: 551568455.3553474}
{1: 0.0, 2: 0.54, 3: 0.37, 4: 0.0, 5: 0.0, 6: 0.0, 7: 0.0, 8: 0.08}
   FUND_ID  ASSET_CLASS_ID  MAX_FEASIBLE_AUM
0        7               1               0.0
1        7

,FUND_ID,MAX_FEASIBLE_AUM
0,9,2.557635e+09
0,7,1.583050e+09
0,12,6.028080e+09
0,10,3.224697e+09
0,11,6.947996e+09


In [40]:
merged_final_df = pd.merge(asset_pool.groupby('FUND_ID')['AVAILABLE_AMOUNT'].sum().reset_index(), final_df , on = 'FUND_ID', how='inner')
merged_final_df['PERC'] = merged_final_df['MAX_FEASIBLE_AUM']/merged_final_df['AVAILABLE_AMOUNT']
merged_final_df

,FUND_ID,AVAILABLE_AMOUNT,MAX_FEASIBLE_AUM,PERC
0,7,6.411761e+09,1.583050e+09,0.246898
1,9,1.553531e+10,2.557635e+09,0.164634
2,10,6.411761e+09,3.224697e+09,0.502935
3,11,2.082664e+10,6.947996e+09,0.333611
4,12,2.082664e+10,6.028080e+09,0.289441
5,13,6.411761e+09,1.554268e+09,0.242409
6,14,2.082664e+10,1.117279e+10,0.536466
7,15,1.066871e+10,3.774109e+09,0.353755


In [41]:
asset_level_df

,FUND_ID,MAX_FEASIBLE_AUM,AVAILABLE_AMOUNT
0,9,2.301872e+08,1.192803e+09
1,9,2.071685e+09,2.071685e+09
2,9,2.046108e+08,3.080218e+09
3,9,0.000000e+00,1.142653e+09
4,9,0.000000e+00,1.339256e+09
...,...,...,...
3,14,1.117279e+08,1.528559e+09
4,14,1.117279e+09,4.110379e+09
5,14,8.938234e+08,4.136494e+09
6,14,1.229007e+09,1.790428e+09


In [47]:
asset_level_df['PERC'] = asset_level_df['MAX_FEASIBLE_AUM']/asset_level_df['AVAILABLE_AMOUNT']

In [48]:
asset_level_df[asset_level_df['FUND_ID']==9]

,FUND_ID,MAX_FEASIBLE_AUM,AVAILABLE_AMOUNT,PERCENT_OF_FUND,PERC
0,9,2.301872e+08,1.192803e+09,0.09,0.192980
1,9,2.071685e+09,2.071685e+09,0.81,1.000000
2,9,2.046108e+08,3.080218e+09,0.08,0.066427
3,9,0.000000e+00,1.142653e+09,0.00,0.000000
4,9,0.000000e+00,1.339256e+09,0.00,0.000000
5,9,0.000000e+00,1.151765e+09,0.00,0.000000
6,9,5.115270e+07,2.496314e+09,0.02,0.020491
7,9,0.000000e+00,3.060619e+09,0.00,0.000000
